In [1]:
import pandas as pd
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

load_dotenv()

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

rfm_query = """
    SELECT
        c.customer_unique_id,
        MAX(o.order_purchase_timestamp) AS last_order_date,
        COUNT(DISTINCT o.order_id) AS frequency,
        SUM(f.price) AS monetary
    FROM dim_orders o
    JOIN dim_customers c ON o.customer_id = c.customer_id
    JOIN fact_order_items f ON o.order_id = f.order_id
    GROUP BY c.customer_unique_id
"""

rfm = pd.read_sql(rfm_query, engine)
rfm.head()

,customer_unique_id,last_order_date,frequency,monetary
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,1,129.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,1,18.90
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,1,69.00
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,1,25.99
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,1,180.00


In [2]:
snapshot_date = rfm["last_order_date"].max() + pd.Timedelta(days=1)
rfm["recency"] = (snapshot_date - rfm["last_order_date"]).dt.days
rfm[["customer_unique_id", "recency", "frequency", "monetary"]].describe()

,recency,frequency,monetary
count,95420.000000,95420.000000,95420.000000
mean,243.600377,1.034018,142.440198
std,153.160320,0.211234,217.656355
min,1.000000,1.000000,0.850000
25%,119.000000,1.000000,47.900000
50%,224.000000,1.000000,89.900000
75%,353.000000,1.000000,155.000000
max,729.000000,16.000000,13440.000000


In [3]:
snapshot_date = rfm["last_order_date"].max() + pd.Timedelta(days=1)
rfm["recency"] = (snapshot_date - rfm["last_order_date"]).dt.days
rfm[["customer_unique_id", "recency", "frequency", "monetary"]].describe()

,recency,frequency,monetary
count,95420.000000,95420.000000,95420.000000
mean,243.600377,1.034018,142.440198
std,153.160320,0.211234,217.656355
min,1.000000,1.000000,0.850000
25%,119.000000,1.000000,47.900000
50%,224.000000,1.000000,89.900000
75%,353.000000,1.000000,155.000000
max,729.000000,16.000000,13440.000000


In [4]:
# Split monetary and recency each into "high" vs "low" using the median as the cutoff
recency_median = rfm["recency"].median()
monetary_median = rfm["monetary"].median()

rfm["recency_flag"] = rfm["recency"].apply(lambda x: "recent" if x <= recency_median else "not_recent")
rfm["monetary_flag"] = rfm["monetary"].apply(lambda x: "high" if x >= monetary_median else "low")

def assign_segment(row):
    if row["recency_flag"] == "recent" and row["monetary_flag"] == "high":
        return "Champions"
    elif row["recency_flag"] == "not_recent" and row["monetary_flag"] == "high":
        return "At Risk"
    elif row["recency_flag"] == "recent" and row["monetary_flag"] == "low":
        return "New / Low Value"
    else:
        return "Lost"

rfm["segment"] = rfm.apply(assign_segment, axis=1)

rfm["segment"].value_counts()

segment
Champions          24147
Lost               24027
New / Low Value    23670
At Risk            23576
Name: count, dtype: int64

In [5]:
segment_summary = rfm.groupby("segment").agg(
    customer_count=("customer_unique_id", "count"),
    avg_recency_days=("recency", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_monetary=("monetary", "mean"),
    total_monetary=("monetary", "sum"),
).round(2)

segment_summary

,customer_count,avg_recency_days,avg_frequency,avg_monetary,total_monetary
segment,,,,,
At Risk,23576,370.46,1.05,238.37,5619728.85
Champions,24147,117.37,1.07,236.63,5713919.05
Lost,24027,370.61,1.01,47.42,1139480.47
New / Low Value,23670,117.09,1.01,47.25,1118515.33


In [6]:
import os
os.makedirs("../06_modeling/outputs", exist_ok=True)
rfm.to_csv("../06_modeling/outputs/customer_rfm_segments.csv", index=False)
segment_summary.to_csv("../06_modeling/outputs/rfm_segment_summary.csv")

In [7]:
model_query = """
    SELECT
        f.order_id,
        f.order_item_id,
        f.price,
        f.freight_value,
        p.product_category_name,
        p.product_weight_g,
        s.seller_state,
        o.order_purchase_timestamp,
        o.order_estimated_delivery_date,
        o.order_delivered_customer_date,
        EXTRACT(DOW FROM o.order_purchase_timestamp) AS purchase_day_of_week
    FROM fact_order_items f
    JOIN dim_products p ON f.product_id = p.product_id
    JOIN dim_sellers s ON f.seller_id = s.seller_id
    JOIN dim_orders o ON f.order_id = o.order_id
    WHERE o.order_delivered_customer_date IS NOT NULL
"""

df = pd.read_sql(model_query, engine)
print(df.shape)
df.head()

(110196, 11)


,order_id,order_item_id,price,freight_value,product_category_name,product_weight_g,seller_state,order_purchase_timestamp,order_estimated_delivery_date,order_delivered_customer_date,purchase_day_of_week
0,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,pet_shop,30000.0,SP,2017-04-26 10:53:06,2017-05-15,2017-05-12 16:04:24,3.0
1,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,moveis_decoracao,3050.0,MG,2018-01-14 14:33:31,2018-02-05,2018-01-22 13:19:16,0.0
2,00048cc3ae777c65dbb7d2a0634bc1ea,1,21.90,12.69,utilidades_domesticas,450.0,SP,2017-05-15 21:42:34,2017-06-06,2017-05-22 13:44:35,1.0
3,0005a1a1728c9d785b8e2b08b904576c,1,145.95,11.65,beleza_saude,2000.0,SP,2018-03-19 18:40:33,2018-03-29,2018-03-29 18:17:31,1.0
4,00061f2a7bc09da83e415a52dc8a4af1,1,59.99,8.88,beleza_saude,950.0,SP,2018-03-24 22:16:10,2018-04-09,2018-03-29 00:04:19,6.0


In [8]:
df["is_late"] = (df["order_delivered_customer_date"] > df["order_estimated_delivery_date"]).astype(int)

print(df["is_late"].value_counts())
print(df["is_late"].value_counts(normalize=True))

# Drop columns that would leak the answer or that we don't need as features
df_model = df.drop(columns=[
    "order_id", "order_item_id",
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
    "order_delivered_customer_date",
])

is_late
0    101481
1      8715
Name: count, dtype: int64
is_late
0    0.920914
1    0.079086
Name: proportion, dtype: float64


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

X = df_model.drop(columns=["is_late"])
y = df_model["is_late"]

# Handle missing product_weight_g (some products may have nulls)
X["product_weight_g"] = X["product_weight_g"].fillna(X["product_weight_g"].median())

categorical_features = ["product_category_name", "seller_state"]
numeric_features = ["price", "freight_value", "product_weight_g", "purchase_day_of_week"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ],
    remainder="passthrough"
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(88156, 6) (22040, 6)
is_late
0    0.920913
1    0.079087
Name: proportion, dtype: float64
is_late
0    0.920917
1    0.079083
Name: proportion, dtype: float64


In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# Build full pipelines: preprocessing + model, chained together
naive_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))
])

balanced_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100, random_state=42, class_weight="balanced"
    ))
])

naive_pipeline.fit(X_train, y_train)
balanced_pipeline.fit(X_train, y_train)

print("Training complete for both models.")

Training complete for both models.


In [12]:
for name, pipeline in [("Naive (no class weighting)", naive_pipeline), ("Balanced (class_weight='balanced')", balanced_pipeline)]:
    preds = pipeline.predict(X_test)
    probs = pipeline.predict_proba(X_test)[:, 1]

    print(f"\n{'=' * 60}")
    print(name)
    print('=' * 60)
    print(classification_report(y_test, preds, target_names=["on_time", "late"]))
    print(f"ROC-AUC: {roc_auc_score(y_test, probs):.3f}")
    print(f"Confusion matrix:\n{confusion_matrix(y_test, preds)}")


Naive (no class weighting)
              precision    recall  f1-score   support

     on_time       0.93      0.99      0.96     20297
        late       0.50      0.15      0.23      1743

    accuracy                           0.92     22040
   macro avg       0.71      0.57      0.59     22040
weighted avg       0.90      0.92      0.90     22040

ROC-AUC: 0.669
Confusion matrix:
[[20031   266]
 [ 1482   261]]

Balanced (class_weight='balanced')
              precision    recall  f1-score   support

     on_time       0.93      0.95      0.94     20297
        late       0.27      0.22      0.24      1743

    accuracy                           0.89     22040
   macro avg       0.60      0.59      0.59     22040
weighted avg       0.88      0.89      0.89     22040

ROC-AUC: 0.673
Confusion matrix:
[[19237  1060]
 [ 1355   388]]


In [15]:
model_query_v2 = """
    SELECT
        f.order_id,
        f.price,
        f.freight_value,
        p.product_category_name,
        p.product_weight_g,
        s.seller_state,
        c.customer_state,
        o.order_purchase_timestamp,
        o.order_estimated_delivery_date,
        o.order_delivered_customer_date,
        EXTRACT(DOW FROM o.order_purchase_timestamp) AS purchase_day_of_week,
        gs.geolocation_lat AS seller_lat,
        gs.geolocation_lng AS seller_lng,
        gc.geolocation_lat AS customer_lat,
        gc.geolocation_lng AS customer_lng
    FROM fact_order_items f
    JOIN dim_products p ON f.product_id = p.product_id
    JOIN dim_sellers s ON f.seller_id = s.seller_id
    JOIN dim_orders o ON f.order_id = o.order_id
    JOIN dim_customers c ON o.customer_id = c.customer_id
    LEFT JOIN dim_geolocation gs ON s.seller_zip_code_prefix = gs.geolocation_zip_code_prefix
    LEFT JOIN dim_geolocation gc ON c.customer_zip_code_prefix = gc.geolocation_zip_code_prefix
    WHERE o.order_delivered_customer_date IS NOT NULL
"""

df2 = pd.read_sql(model_query_v2, engine)
print(df2.shape)
df2.isnull().sum()

(110196, 15)


order_id                            0
price                               0
freight_value                       0
product_category_name            1537
product_weight_g                   18
seller_state                        0
customer_state                      0
order_purchase_timestamp            0
order_estimated_delivery_date       0
order_delivered_customer_date       0
purchase_day_of_week                0
seller_lat                        249
seller_lng                        249
customer_lat                      288
customer_lng                      288
dtype: int64

In [17]:
import numpy as np

# Drop rows missing geolocation or category - can't build features without them
df2 = df2.dropna(subset=["seller_lat", "seller_lng", "customer_lat", "customer_lng", "product_category_name"])

# Feature 1: promised delivery window, in days
df2["estimated_delivery_days"] = (
    df2["order_estimated_delivery_date"] - df2["order_purchase_timestamp"]
).dt.days

# Feature 2: haversine distance between seller and customer, in km
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371  # Earth's radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

df2["seller_customer_distance_km"] = haversine_km(
    df2["seller_lat"], df2["seller_lng"], df2["customer_lat"], df2["customer_lng"]
)

df2["is_late"] = (df2["order_delivered_customer_date"] > df2["order_estimated_delivery_date"]).astype(int)

print(df2.shape)
df2[["estimated_delivery_days", "seller_customer_distance_km", "is_late"]].describe()

(108129, 18)


,estimated_delivery_days,seller_customer_distance_km,is_late
count,108129.000000,108129.000000,108129.000000
mean,23.442601,596.696593,0.078786
std,8.845921,588.548482,0.269405
min,2.000000,0.000000,0.000000
25%,18.000000,186.696894,0.000000
50%,23.000000,432.351740,0.000000
75%,28.000000,791.644192,0.000000
max,155.000000,8677.911641,1.000000


In [25]:
X2 = df2[[
    "price", "freight_value", "product_weight_g",
    "purchase_day_of_week", "estimated_delivery_days",
    "seller_customer_distance_km",
    "product_category_name", "seller_state", "customer_state"
]].copy()
y2 = df2["is_late"]

categorical_features_v2 = ["product_category_name", "seller_state", "customer_state"]

preprocessor_v2 = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features_v2),
    ],
    remainder="passthrough"
)

# Fill the missing product_weight_g values with the median
X2["product_weight_g"] = X2["product_weight_g"].fillna(X2["product_weight_g"].median())

# Confirm no NaNs remain anywhere
print("Null check after fill:")
print(X2.isnull().sum())


Null check after fill:
price                          0
freight_value                  0
product_weight_g               0
purchase_day_of_week           0
estimated_delivery_days        0
seller_customer_distance_km    0
product_category_name          0
seller_state                   0
customer_state                 0
dtype: int64


In [26]:
# Re-split (same split logic as before, now on the cleaned X2)
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

# Rebuild and fit the pipeline
balanced_pipeline_v2 = Pipeline([
    ("preprocessor", preprocessor_v2),
    ("classifier", RandomForestClassifier(
        n_estimators=100, random_state=42, class_weight="balanced"
    ))
])

balanced_pipeline_v2.fit(X2_train, y2_train)

# Evaluate
preds_v2 = balanced_pipeline_v2.predict(X2_test)
probs_v2 = balanced_pipeline_v2.predict_proba(X2_test)[:, 1]

print("\nClassification report:")
print(classification_report(y2_test, preds_v2, target_names=["on_time", "late"]))
print(f"ROC-AUC: {roc_auc_score(y2_test, probs_v2):.3f}")
print(f"Confusion matrix:\n{confusion_matrix(y2_test, preds_v2)}")


Classification report:
              precision    recall  f1-score   support

     on_time       0.94      0.98      0.96     19922
        late       0.52      0.24      0.33      1704

    accuracy                           0.92     21626
   macro avg       0.73      0.61      0.64     21626
weighted avg       0.90      0.92      0.91     21626

ROC-AUC: 0.769
Confusion matrix:
[[19546   376]
 [ 1297   407]]
